# Vector search

## Setup AI Search and OpenAI clients

In [1]:
import os

from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from dotenv import load_dotenv
from openai import OpenAI

from render_table import render_product_results

load_dotenv()

openai_client = OpenAI(
    base_url=f"https://{os.environ['AZURE_OPENAI_SERVICE']}.openai.azure.com/openai/v1",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

def get_embedding_vector(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"],
        input=[query],
    )
    embedding_vector = response.data[0].embedding
    return embedding_vector

search_client = SearchClient(
    f"https://{os.environ['AZURE_SEARCH_SERVICE']}.search.windows.net",
    "zava-products-index",
    credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
)


## Perform vector search

In [2]:
search_query = "water plants efficiently without waste"
search_vector = get_embedding_vector(search_query)

results = search_client.search(
    None, top=5, vector_queries=[VectorizedQuery(vector=search_vector, k_nearest_neighbors=50, fields="embedding")]
)

render_product_results(results)

┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃  Score ┃ Name                    ┃ Description                                     ┃ Categories         ┃ Price ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│  0.626 │ Self-Watering Planter   │ Smart planter with built-in water reservoir for │ GARDEN & OUTDOOR,  │ $35.… │
│        │                         │ consistent plant hydration.                     │ PLANTERS           │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.619 │ Soaker Hose 25-foot     │ Porous soaker hose for efficient water delivery │ GARDEN & OUTDOOR,  │ $25.… │
│        │                         │ directly to plant roots.                        │ HOSES              │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.591 │ Water-Based Wood Stain  │ Fast-drying water-based stain with low odor and │ PAINT & FINISHES,  │ $27.… │
│        │                         │ easy cleanup while maintaining w...             │ WOOD STAIN         │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.591 │ Expandable Hose         │ Lightweight expandable hose that extends when   │ GARDEN & OUTDOOR,  │ $50.… │
│        │ 100-foot                │ water flows and contracts when emp...           │ HOSES              │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.582 │ Hydro Jet Drain Cleaner │ High-pressure water jet attachment for clearing │ PLUMBING, DRAIN    │ $42.… │
│        │                         │ stubborn drain clogs.                           │ CLEANING           │       │
└────────┴─────────────────────────┴─────────────────────────────────────────────────┴────────────────────┴───────┘

## Perform vector search #2

In [3]:
search_query = "100 foot hose that wont break"
search_vector = get_embedding_vector(search_query)

results = search_client.search(
    None, top=5, vector_queries=[VectorizedQuery(vector=search_vector, k_nearest_neighbors=50, fields="embedding")]
)

render_product_results(results)

┏━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃  Score ┃ Name                    ┃ Description                                     ┃ Categories         ┃ Price ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│  0.713 │ Heavy Duty Hose 75-foot │ Commercial-grade garden hose for heavy use with │ GARDEN & OUTDOOR,  │ $58.… │
│        │                         │ reinforced construction.                        │ HOSES              │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.710 │ Garden Hose 50-foot     │ Heavy-duty garden hose with brass fittings and  │ GARDEN & OUTDOOR,  │ $32.… │
│        │                         │ kink-resistant construction.                    │ HOSES              │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.706 │ Expandable Hose         │ Lightweight expandable hose that extends when   │ GARDEN & OUTDOOR,  │ $50.… │
│        │ 100-foot                │ water flows and contracts when emp...           │ HOSES              │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.672 │ Soaker Hose 25-foot     │ Porous soaker hose for efficient water delivery │ GARDEN & OUTDOOR,  │ $25.… │
│        │                         │ directly to plant roots.                        │ HOSES              │       │
├────────┼─────────────────────────┼─────────────────────────────────────────────────┼────────────────────┼───────┤
│  0.645 │ Drinking Water Safe     │ Lead-free garden hose safe for drinking water   │ GARDEN & OUTDOOR,  │ $40.… │
│        │ Hose                    │ and filling pools.                              │ HOSES              │       │
└────────┴─────────────────────────┴─────────────────────────────────────────────────┴────────────────────┴───────┘